In [ ]:
import pandas as pd

In [ ]:
import os

base_path = "/content/data"
os.listdir(base_path)


In [ ]:
!unzip -q /content/archive.zip -d /content/


In [ ]:
!apt-get install unrar


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
# !unrar x /content/data.rar /content/


In [ ]:
import os
os.listdir("/content")


['.config',
 'data',
 'label_map.json',
 'data.rar',
 'news_classifier_model',
 'results',
 'sample_data']

In [ ]:
os.listdir("/content/data")


['Sports', 'Tech', 'Culture', 'Medical', 'Finance', 'Politics', 'Religion']

In [ ]:
import os

category = "Politics"   # غيّرها لو تحب
file_name = os.listdir(f"/content/data/{category}")[0]

with open(f"/content/data/{category}/{file_name}", encoding="utf-8") as f:
    text = f.read()

print(text[:400])


كتب:عمار عوضتناغمت آراء الصحافة العالمية مع مبايعة الأمير محمد بن سلمان ولياً للعهد في السعودية، واعتبرت صحيفة «وول ستريت جورنال» الهدف الأول للتغييرات في السعودية هو نقل البلاد إلى العصر الحديث، مؤكدة أنه سيكون لهذا الأمر انعكاسات عميقة على المستقبل السياسي والاقتصادي للمملكة، وأسواق النفط العالمية، وحلفائها في الشرق الأوسط، وخارجه، بينما رأت صحيفة «ديلي ميل» البريطانية أن التغيرات ينظر لها في ال


In [ ]:
import os
import pandas as pd

base_path = "/content/data"

data = []

for label in os.listdir(base_path):
    folder_path = os.path.join(base_path, label)

    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)

        with open(file_path, encoding="utf-8") as f:
            text = f.read()

        data.append({
            "text": text,
            "label": label
        })

df = pd.DataFrame(data)


In [ ]:
import re

def arabert_clean(text):
    # إزالة التشكيل
    text = re.sub(r'[\u064B-\u065F]', '', text)

    # إزالة التطويل
    text = re.sub(r'ـ+', '', text)

    # حذف أي شيء غير عربي (نترك المسافات)
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    # توحيد المسافات
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [ ]:
sample_text = df['text'].iloc[0]
cleaned = arabert_clean(sample_text)



قبل:
 الشارقة: ضمياء فالح طالب نادي إشبيلية الإسباني مديره الرياضي مونشي (47 عاماً) بدفع مبلغ 3.8 مليون جنيه إسترليني شرط العقد الجزائي معه، لأن النادي الأندلسي يشك في أن «مستر موني بول» لديه عروض من مانشستر يونايتد وريال مدريد.وكان مونشي أبلغ إدارة إشبيلية أنه يريد ترك العمل فيه «لأسباب شخصية». وشارك مون

بعد:
 الشارقة ضمياء فالح طالب نادي إشبيلية الإسباني مديره الرياضي مونشي عاما بدفع مبلغ مليون جنيه إسترليني شرط العقد الجزائي معه، لأن النادي الأندلسي يشك في أن مستر موني بول لديه عروض من مانشستر يونايتد وريال مدريد وكان مونشي أبلغ إدارة إشبيلية أنه يريد ترك العمل فيه لأسباب شخصية وشارك مونشي في قيادة الفر


In [ ]:
df.head()
df.shape
df['label'].value_counts()


,count
label,
Sports,6500
Tech,6500
Culture,6500
Medical,6500
Finance,6500
Politics,6500
Religion,6500


In [ ]:
def remove_author(text):
    # إزالة اسم الكاتب إذا كان في بداية النص
    text = re.sub(r'^[^:\n]{1,40}:\s*', '', text)
    return text


In [ ]:
def full_clean_stage1(text):
    text = arabert_clean(text)
    text = remove_author(text)
    return text


In [ ]:
test_clean = full_clean_stage1(df['text'].iloc[0])

print("بعد التنظيف النهائي:\n", test_clean[:400])


بعد التنظيف النهائي:
 الشارقة ضمياء فالح طالب نادي إشبيلية الإسباني مديره الرياضي مونشي عاما بدفع مبلغ مليون جنيه إسترليني شرط العقد الجزائي معه، لأن النادي الأندلسي يشك في أن مستر موني بول لديه عروض من مانشستر يونايتد وريال مدريد وكان مونشي أبلغ إدارة إشبيلية أنه يريد ترك العمل فيه لأسباب شخصية وشارك مونشي في قيادة الفريق إلى ألقاب في سنوات وحقق مليون جنيه إسترليني أرباحا في صفقات بيع اللاعبين وتعتبره الإدارة عنصرا مه


In [ ]:
df['clean_text'] = df['text'].apply(full_clean_stage1)


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df['label'])

df[['label', 'label_id']].head()


,label,label_id
0,Sports,5
1,Sports,5
2,Sports,5
3,Sports,5
4,Sports,5


In [ ]:
label_encoder.classes_


array(['Culture', 'Finance', 'Medical', 'Politics', 'Religion', 'Sports',
       'Tech'], dtype=object)

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label_id']

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)


In [ ]:
print("Train distribution:\n", y_train.value_counts().sort_index())
print("\nVal distribution:\n", y_val.value_counts().sort_index())
print("\nTest distribution:\n", y_test.value_counts().sort_index())


Train distribution:
 label_id
0    5200
1    5200
2    5200
3    5200
4    5200
5    5200
6    5200
Name: count, dtype: int64

Val distribution:
 label_id
0    650
1    650
2    650
3    650
4    650
5    650
6    650
Name: count, dtype: int64

Test distribution:
 label_id
0    650
1    650
2    650
3    650
4    650
5    650
6    650
Name: count, dtype: int64


In [ ]:
# !pip install -U huggingface_hub


In [ ]:
# !huggingface-cli login


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "UBC-NLP/MARBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("MARBERT tokenizer loaded successfully")


MARBERT tokenizer loaded successfully


In [ ]:

train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "text": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )


In [ ]:
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)


Map:   0%|          | 0/36400 [00:00<?, ? examples/s]

Map:   0%|          | 0/4550 [00:00<?, ? examples/s]

Map:   0%|          | 0/4550 [00:00<?, ? examples/s]

In [ ]:
train_ds[0]


{'text': 'حققت حملة متاجر شرف دي جي وشركة هيسنس مبيعات ونجاحا كبيرين بسبب العرض الكبير الذي أطلقته خلال المعرض وهو اشتر جهاز تلفاز هيسنس بوصة واحصل على سيارة بي إم دبليو أي موديل وقام متسوقين خلال اليوم الأول من جيتكس شوبر بشراء تلفاز هيسنس بوصة عالي الدقة من شرف دي جي، حيث تمت دعوتهم لاستلام شهادات الملكية الخاصه بسياراتهم الجديدة من بي إم دبليو طراز أي موديل وقال عيسى علي أهلي أحد الفائزين بالسيارة هذا العرض لا يصدق ولقد حصلت على أكثر مما كنت أتوقع، ومن جهته قال شيخ عمران هذه الصفقة رائعة ويجب أن لا تفوت، في ما أوضح تينجري بال سان إنه عرض لا يصدق، لم يسبق لي أن سمعت بعرض مماثل من قبل وقال ياسر شرف، المدير الإداري لمتاجر شرف دي جي إن الشركة عودت المتسوقين خلال جيتكس شوبر على عروض أسعار غير مسبوقة وتخفيضات حقيقية،وذلك من خلال رصد ميزانية لهذه العروض والتخفيضات تقدر بملايين الدراهم',
 'label': 6,
 '__index_level_0__': 10513,
 'input_ids': [2,
  20890,
  5069,
  53174,
  5052,
  2441,
  3696,
  53754,
  24425,
  3905,
  33921,
  27685,
  1011,
  3102,
  1943,
  3463,
  5960,
  4935,
  22

In [ ]:
from transformers import AutoModelForSequenceClassification

NUM_LABELS = 7

model = AutoModelForSequenceClassification.from_pretrained(
    "UBC-NLP/MARBERT",
    num_labels=NUM_LABELS
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

In [ ]:
# !pip install -U transformers datasets accelerate


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    report_to="none"
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.train(resume_from_checkpoint=True)


In [ ]:
test_metrics = trainer.evaluate(test_ds)
test_metrics


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred))


In [ ]:
from transformers import AutoModelForSequenceClassification

best_model_path = "./results/checkpoint-9100"

best_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)

best_model.save_pretrained("./news_classifier_model")
tokenizer.save_pretrained("./news_classifier_model")


In [ ]:
import json

label_map = {
    label: int(idx)
    for label, idx in zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
}

with open("label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)


In [ ]:
def predict_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = best_model(**inputs)
    pred_id = outputs.logits.argmax(dim=1).item()
    return label_encoder.inverse_transform([pred_id])[0]

predict_news("فاز المنتخب الوطني في المباراة النهائية للبطولة")


In [ ]:
!cat label_map.json


In [ ]:
predict_news("أعلنت وزارة المالية عن الموازنة العامة الجديدة")
# متوقع: Finance



In [ ]:
predict_news("اكتشف الأطباء علاجًا جديدًا لمرض نادر")
# متوقع: Medical



In [ ]:
predict_news("ناقش البرلمان مشروع قانون جديد")
# متوقع: Politics


'Politics'

In [ ]:
!ls


data  data.rar	label_map.json	news_classifier_model  results	sample_data


In [ ]:
!zip -r news_classifier_model.zip news_classifier_model label_map.json


  adding: news_classifier_model/ (stored 0%)
  adding: news_classifier_model/config.json (deflated 59%)
  adding: news_classifier_model/model.safetensors (deflated 7%)
  adding: news_classifier_model/tokenizer_config.json (deflated 45%)
  adding: news_classifier_model/tokenizer.json (deflated 72%)
  adding: label_map.json (deflated 23%)


In [ ]:
from google.colab import files
files.download("news_classifier_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
best_model
tokenizer
label_map


{'Culture': 0,
 'Finance': 1,
 'Medical': 2,
 'Politics': 3,
 'Religion': 4,
 'Sports': 5,
 'Tech': 6}

In [ ]:
import json

with open("label_map.json", encoding="utf-8") as f:
    label_map = json.load(f)

inv_label_map = {int(v): k for k, v in label_map.items()}

print(inv_label_map)


{0: 'Culture', 1: 'Finance', 2: 'Medical', 3: 'Politics', 4: 'Religion', 5: 'Sports', 6: 'Tech'}


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import torch

app = FastAPI(title="Arabic News Classifier API")


In [ ]:
class NewsRequest(BaseModel):
    text: str


In [ ]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # للتجربة فقط
    allow_credentials=True,
    allow_methods=["*"],   # مهم جدًا
    allow_headers=["*"],
)


In [ ]:
ngrok.kill()

In [ ]:
@app.post("/predict")
def predict_news(req: NewsRequest):
    try:
        if not req.text or len(req.text.strip()) < 5:
            return {"label": "Unknown"}

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        best_model.to(device)

        inputs = tokenizer(
            req.text,
            return_tensors="pt",
            truncation=True,
            padding=True
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = best_model(**inputs)

        pred_id = outputs.logits.argmax(dim=1).item()

        label = inv_label_map.get(pred_id, "Unknown")

        return {"label": label}

    except Exception as e:
        print("❌ API ERROR:", e)
        return {"error": str(e)}


In [ ]:
@app.get("/")
def root():
    return {"status": "API is running"}


In [ ]:
!pip install pyngrok


In [ ]:
import nest_asyncio
nest_asyncio.apply()


In [ ]:
import threading
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()



INFO:     Started server process [14295]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("37qNzng4PGphD7Sdi5flitFYojX_2R96tZ8GD54Y1yBjXUJUf")

In [ ]:
public_url = ngrok.connect(8000)
print(public_url)


NgrokTunnel: "https://paraglossate-untrapped-tena.ngrok-free.dev" -> "http://localhost:8000"
